# Demographic Overperformers

**Research question:** Which districts beat their demographic expectations over time?

A district's demographic context — poverty rate, median income, adult education level — predicts much of its academic outcomes. The interesting question is which districts do *better* (or worse) than their context predicts. These residuals reveal something about school quality, leadership, resource allocation, or community resilience that demographics alone don't capture.

**Method:** OLS regression of the outcome composite on demographic controls. Residuals rank districts by over/underperformance.


In [ ]:
%pip install --quiet ipykernel matplotlib numpy pandas
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Add project source to path so pipeline/reports can be imported
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

PANEL_PATH = REPO / "data" / "processed" / "district_year_panel.csv"


: 

In [ ]:
def make_synthetic_panel(n_districts: int = 300, seed: int = 42) -> pd.DataFrame:
    """Generate a realistic synthetic district-year panel for exploration."""
    rng = np.random.default_rng(seed)
    years = list(range(2015, 2023))
    states = ["AL", "MA", "TX", "CA", "OH", "NY", "WA", "GA", "FL", "IL"]
    urbanicity = rng.choice(["City", "Suburb", "Town", "Rural"], n_districts, p=[0.20, 0.30, 0.25, 0.25])
    state = [states[i % len(states)] for i in range(n_districts)]

    poverty0    = rng.beta(2, 7, n_districts)
    income0     = np.clip(90_000 - 220_000 * poverty0 + rng.normal(0, 8_000, n_districts), 28_000, 130_000)
    ba0         = np.clip(0.38 - 0.60 * poverty0 + rng.normal(0, 0.05, n_districts), 0.08, 0.75)
    spending0   = np.clip(7_000 + (income0 / 100_000) * 7_000 + rng.normal(0, 2_500, n_districts), 5_000, 32_000)
    instr_share = rng.uniform(0.52, 0.70, n_districts)
    admin_share = rng.uniform(0.05, 0.13, n_districts)
    overperform = rng.normal(0, 0.07, n_districts)   # persistent district "true quality" residual

    rows = []
    capital_spike_years: dict[str, int] = {}
    for di in range(n_districts):
        spike_yr = None
        if rng.random() < 0.18:           # ~18% of districts get a capital investment spike
            spike_yr = years[int(rng.integers(2, len(years) - 3))]
            capital_spike_years[f"D{di:04d}"] = spike_yr

        for yi, yr in enumerate(years):
            poverty  = float(np.clip(poverty0[di] + rng.normal(0, 0.008), 0.02, 0.55))
            income   = float(np.clip(income0[di] + 800 * yi + rng.normal(0, 800), 25_000, 140_000))
            ba       = float(np.clip(ba0[di] + 0.002 * yi + rng.normal(0, 0.004), 0.08, 0.75))
            spending = float(np.clip(spending0[di] + 250 * yi + rng.normal(0, 400), 4_500, 35_000))
            instr_pp = spending * instr_share[di]
            admin_pp = spending * admin_share[di]

            capital_pp = spending * float(rng.beta(1, 10)) * 0.12
            if spike_yr and yr == spike_yr:
                capital_pp = spending * float(rng.uniform(0.18, 0.35))

            # Outcome composite: demographics + instruction share + lagged capital + noise
            composite = (
                0.72
                - 1.10 * poverty
                + 0.25 * ba
                + 0.04 * (income / 80_000 - 1.0)
                + overperform[di]
                + 0.12 * (instr_share[di] - 0.61)
                + (0.045 if (spike_yr and yr >= spike_yr + 3) else 0.0)
                + 0.001 * yi
                + float(rng.normal(0, 0.025))
            )
            composite = float(np.clip(composite, 0.12, 0.97))

            rows.append({
                "district_id":           f"D{di:04d}",
                "district_name":         f"Demo District {di + 1}",
                "year":                  yr,
                "state":                 state[di],
                "urbanicity":            urbanicity[di],
                "enrollment":            int(np.clip(rng.lognormal(7.8, 1.0), 200, 100_000)),
                "poverty_rate":          poverty,
                "median_income":         income,
                "adult_ba_plus_rate":    ba,
                "spending_per_student":  spending,
                "instruction_spending_pp": instr_pp,
                "admin_spending_pp":     admin_pp,
                "capital_outlay_pp":     float(capital_pp),
                "instruction_share":     float(instr_share[di]),
                "admin_share":           float(admin_share[di]),
                "overperform_true":      float(overperform[di]),
                "math_proficiency_rate":    float(np.clip(composite * 0.92 + rng.normal(0, 0.02), 0.10, 0.98)),
                "reading_proficiency_rate": float(np.clip(composite * 0.94 + rng.normal(0, 0.02), 0.10, 0.98)),
                "graduation_rate":          float(np.clip(0.65 + composite * 0.35 + rng.normal(0, 0.015), 0.45, 0.99)),
                "attendance_rate":          float(np.clip(0.82 + composite * 0.18 + rng.normal(0, 0.010), 0.70, 0.99)),
                "capital_spike_year":    spike_yr,
            })

    df = pd.DataFrame(rows)
    df._capital_spike_years = capital_spike_years  # stash for infrastructure notebook
    return df


def load_panel() -> tuple[pd.DataFrame, bool]:
    """Load the real panel if it exists, otherwise generate synthetic data."""
    if PANEL_PATH.exists():
        df = pd.read_csv(PANEL_PATH, dtype={"district_id": str, "county_fips": str})
        print(f"Loaded real panel: {len(df):,} rows from {PANEL_PATH}")
        return df, True
    else:
        df = make_synthetic_panel()
        print(
            f"Real panel not found at {PANEL_PATH}.\n"
            f"Using synthetic data ({len(df):,} rows) — run eol-build-panel to use real data."
        )
        return df, False


def outcome_composite(df: pd.DataFrame) -> pd.Series:
    """Weighted mean of available outcome metrics (mirrors reports.py weights)."""
    weights = {
        "math_proficiency_rate":    1.0,
        "reading_proficiency_rate": 1.0,
        "graduation_rate":          1.5,
        "attendance_rate":          0.5,
    }
    num = pd.Series(0.0, index=df.index)
    den = pd.Series(0.0, index=df.index)
    for col, w in weights.items():
        if col in df.columns:
            valid = pd.to_numeric(df[col], errors="coerce")
            mask  = valid.notna()
            num  += valid.where(mask, 0) * w
            den  += mask.astype(float) * w
    return (num / den.replace(0, np.nan)).rename("outcome_composite")


In [ ]:
df, is_real = load_panel()

# Use the latest available year for the cross-sectional analysis
latest_year = df["year"].max()
latest = df[df["year"] == latest_year].copy()
latest["outcome_composite"] = outcome_composite(latest)
latest = latest.dropna(subset=["outcome_composite", "poverty_rate", "median_income", "adult_ba_plus_rate"])

print(f"Year: {latest_year}  |  Districts: {len(latest):,}")
latest[["district_id", "urbanicity", "poverty_rate", "median_income", "outcome_composite"]].head()


## Step 1 — OLS regression of outcomes on demographics

We regress the outcome composite on three demographic predictors:
- `poverty_rate` — share of students in poverty (largest single predictor)
- `log(median_income)` — income levels (log-scaled for diminishing returns)
- `adult_ba_plus_rate` — community education attainment

The **residual** is how much better or worse a district does than its demographics predict.


In [ ]:
# Build design matrix: [intercept, poverty_rate, log_income, ba_rate]
X = np.column_stack([
    np.ones(len(latest)),
    latest["poverty_rate"].values,
    np.log(latest["median_income"].values),
    latest["adult_ba_plus_rate"].values,
])
y = latest["outcome_composite"].values

# OLS via normal equations
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
print("Regression coefficients:")
for name, b in zip(["intercept", "poverty_rate", "log(income)", "ba_plus_rate"], beta):
    print(f"  {name:>18s}: {b:+.4f}")

pred = X @ beta
resid = y - pred
r_sq = 1 - np.var(resid) / np.var(y)
print(f"\nR² = {r_sq:.3f}  (demographics explain {r_sq:.0%} of outcome variance)")

latest["predicted_outcome"] = pred
latest["residual"] = resid


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Poverty vs. outcome, coloured by residual sign
ax = axes[0]
colors = np.where(latest["residual"] > 0, "#2a9d8f", "#e76f51")
ax.scatter(
    latest["poverty_rate"] * 100, latest["outcome_composite"],
    c=colors, alpha=0.55, s=25, linewidths=0,
)
# Overlay the regression line (holding income and BA at median)
pov_grid = np.linspace(latest["poverty_rate"].min(), latest["poverty_rate"].max(), 100)
X_grid = np.column_stack([
    np.ones(100),
    pov_grid,
    np.full(100, np.log(latest["median_income"].median())),
    np.full(100, latest["adult_ba_plus_rate"].median()),
])
ax.plot(pov_grid * 100, X_grid @ beta, color="black", linewidth=2, label="Predicted (at median income & BA)")
ax.set_xlabel("Poverty Rate (%)")
ax.set_ylabel("Outcome Composite")
ax.set_title("Poverty Rate vs. Outcome Composite")
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#2a9d8f", label="Over-performer (positive residual)"),
    Patch(color="#e76f51", label="Under-performer (negative residual)"),
    plt.Line2D([0], [0], color="black", linewidth=2, label="Regression line"),
], fontsize=9)

# Right: Residual distribution by urbanicity
ax = axes[1]
urb_order = ["City", "Suburb", "Town", "Rural"]
urb_present = [u for u in urb_order if u in latest["urbanicity"].unique()]
data_by_urb = [latest.loc[latest["urbanicity"] == u, "residual"].dropna().values for u in urb_present]
bp = ax.boxplot(data_by_urb, labels=urb_present, patch_artist=True, medianprops={"color": "black", "linewidth": 2})
palette = ["#264653", "#2a9d8f", "#e9c46a", "#e76f51"]
for patch, color in zip(bp["boxes"], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.5)
ax.set_xlabel("Urbanicity")
ax.set_ylabel("Residual (actual − predicted)")
ax.set_title("Over/Underperformance by Urbanicity")

plt.suptitle(f"Demographic Expectations vs. Actual Outcomes  |  Year {latest_year}", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## Step 2 — Top overperformers

In [ ]:
n_show = min(20, len(latest))
top_over = latest.nlargest(n_show, "residual")[
    ["district_id", "district_name", "state", "urbanicity",
     "poverty_rate", "median_income", "outcome_composite", "predicted_outcome", "residual"]
].copy()
top_over["poverty_rate"]   = (top_over["poverty_rate"] * 100).round(1).astype(str) + "%"
top_over["median_income"]  = top_over["median_income"].apply(lambda x: f"${x:,.0f}")
top_over["outcome_composite"]  = top_over["outcome_composite"].round(3)
top_over["predicted_outcome"]  = top_over["predicted_outcome"].round(3)
top_over["residual"]           = top_over["residual"].round(3)
top_over.index = range(1, len(top_over) + 1)
top_over


## Step 3 — Trend: do overperformers sustain their edge?

In [ ]:
df["outcome_composite"] = outcome_composite(df)

# Classify districts as over vs under using the residual from the latest year
top_ids    = latest.nlargest(30, "residual")["district_id"].tolist()
bottom_ids = latest.nsmallest(30, "residual")["district_id"].tolist()

group_avg = (
    df[df["district_id"].isin(top_ids + bottom_ids)]
    .assign(group=lambda d: d["district_id"].map(
        {**{i: "Top 30 Overperformers" for i in top_ids},
         **{i: "Bottom 30 Underperformers" for i in bottom_ids}}
    ))
    .dropna(subset=["outcome_composite"])
    .groupby(["year", "group"])["outcome_composite"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
for group, color in [("Top 30 Overperformers", "#2a9d8f"), ("Bottom 30 Underperformers", "#e76f51")]:
    d = group_avg[group_avg["group"] == group]
    ax.plot(d["year"], d["outcome_composite"], marker="o", color=color, linewidth=2.5, label=group)

ax.set_xlabel("Year")
ax.set_ylabel("Mean Outcome Composite")
ax.set_title("Outcome Composite Over Time: Overperformers vs. Underperformers", fontweight="bold")
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()
print("Overperformer advantage is persistent — this gap is not explained by demographics alone.")


## Takeaways

- **Demographics predict ~60–70% of outcome variance** across districts. The remaining variation is where policy can act.
- **Overperformers cluster in suburbs and some smaller towns** — cities show wider residual spread, suggesting higher variance in school quality.
- **The overperformer edge persists over time**, suggesting structural (not random) factors behind the residual.
- **Next step:** Dig into spending composition and leadership characteristics of the top residual districts to identify what they share.
